In [1]:
# mypy: ignore-errors
# ruff: noqa
import gc
import math
import os
import sys
import warnings

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

warnings.simplefilter("ignore")
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# ruff: noqa
%load_ext autoreload
%autoreload 2
from drGAT import drGAT
from drGAT.load_data import load_data
from drGAT.sampler import NewSampler
from get_params import get_params
from metrics import compute_metrics_stats

In [3]:
name = "nci"

In [4]:
(
    res,
    pos_num,
    null_mask,
    drug_sim,
    cell_sim,
    gene_sim,
    A_cg,
    A_dg,
    _,
    _,
    _,
) = load_data(name)
res = res.T
cell_sum = np.sum(res, axis=1)
drug_sum = np.sum(res, axis=0)

target_dim = [
    0,  # Cell
    # 1  # Drug
]

load nci
unique drugs: 177
unique genes: 251
DTI unique genes:  251
Top 90% variable genes:  2383
Total:  2582
Final gene exp shape: (60, 2582)
Final drug Act shape: (1005, 60)


100%|██████████| 25/25 [00:02<00:00,  8.74it/s]


Done!


In [5]:
drug_sum

740       38
752       35
755       34
757       39
762       26
          ..
811429    25
812926    16
812927    23
813488    24
820919    25
Length: 1005, dtype: int64

In [6]:
def drGAT_new(
    res_mat,
    null_mask,
    target_dim,
    target_index,
    S_d,
    S_c,
    S_g,
    A_cg,
    A_dg,
    PATH,
    params,
    device,
    seed,
):
    sampler = NewSampler(
        res_mat,
        null_mask,
        target_dim,
        target_index,
        S_d,
        S_c,
        S_g,
        A_cg,
        A_dg,
        PATH,
        seed,
    )

    (_, _, _, best_val_labels, best_val_prob, best_metrics, _, _, _) = drGAT.train(
        sampler, params=params, device=device, verbose=False
    )

    return best_val_labels, best_val_prob

In [7]:
method = "GAT"
params = get_params(method, f"NCI_{method}_New")
PATH = f"../{name}_data/"
params.update(
    {
        "n_drug": drug_sim.shape[0],
        "n_cell": cell_sim.shape[0],
        "n_gene": gene_sim.shape[0],
        "epochs": 1,
        "params_heads": 1,
        "params_hidden1": 128,
    }
)

[I 2025-04-08 17:28:11,698] Using an existing study with name 'NCI_GAT_New' instead of creating a new one.


number                                            137
values_0                                     0.658982
values_1                                     0.727157
values_2                                     0.730925
values_3                                     0.609317
datetime_start             2025-04-01 12:35:37.260208
datetime_complete          2025-04-01 13:33:22.498690
duration                       0 days 00:57:45.238482
params_T_max                                      NaN
params_activation                                gelu
params_amsgrad                                  False
params_dropout1                                   0.3
params_dropout2                                   0.3
params_epochs                                   546.0
params_gamma_step                                 NaN
params_gnn_layer                                  GAT
params_heads                                      5.0
params_hidden1                                 1028.0
params_hidden2              

In [8]:
n_kfold = 1
true_datas = pd.DataFrame()
predict_datas = pd.DataFrame()
for dim in target_dim:
    for seed, target_index in tqdm(enumerate(np.arange(res.shape[dim]))):
        if dim:
            if drug_sum[target_index] < 10:
                continue
        else:
            if cell_sum[target_index] < 10:
                continue
        epochs = []
        true_data_s = pd.DataFrame()
        predict_data_s = pd.DataFrame()
        for fold in range(n_kfold):
            true_data, predict_data = drGAT_new(
                res_mat=res,
                null_mask=null_mask.T.values,
                target_dim=dim,
                target_index=target_index,
                S_d=drug_sim,
                S_c=cell_sim,
                S_g=gene_sim,
                A_cg=A_cg,
                A_dg=A_dg,
                PATH=PATH,
                params=params,
                device=device,
                seed=seed,
            )

            true_datas = pd.concat(
                [true_datas, pd.DataFrame(true_data).T], ignore_index=True
            )
            predict_datas = pd.concat(
                [predict_datas, pd.DataFrame(predict_data).T], ignore_index=True
            )

0it [00:00, ?it/s]

Using device: cuda


1it [00:04,  4.03s/it]

Best model found at epoch 1
Using device: cuda


2it [00:06,  3.18s/it]

...

Best model found at epoch 1
Using device: cuda


59it [03:02,  3.60s/it]

Best model found at epoch 1
Using device: cuda


60it [03:06,  3.11s/it]

Best model found at epoch 1


In [9]:
# true_datas.to_csv(f"new_true_{method}_{args.data}.csv")
# predict_datas.to_csv(f"new_pred_{method}_{args.data}.csv")

In [12]:
# true_datas

In [13]:
np.all(
    true_datas.values
    == pd.read_csv("../../DRBenchmark/NIHGCN/new_cell_true_nci.csv", index_col=0).values
)

False

In [14]:
true_datas.dropna().values[0] == pd.read_csv(
    "../../DRBenchmark/NIHGCN/new_cell_true_nci.csv", index_col=0
).dropna().values[0]

array([ True,  True, False, ..., False,  True, False])

In [15]:
true_datas.dropna().values[0]

array([1., 1., 0., ..., 0., 0., 0.])

In [16]:
pd.read_csv(
    "../../DRBenchmark/NIHGCN/new_cell_true_nci.csv", index_col=0
).dropna().values[0]

array([1., 1., 1., ..., 1., 0., 1.])

In [44]:
sampler = NewSampler(
    res,
    null_mask=null_mask.T.values,
    target_dim=dim,
    target_index=target_index,
    S_d=drug_sim,
    S_c=cell_sim,
    S_g=gene_sim,
    A_cg=A_cg,
    A_dg=A_dg,
    PATH=PATH,
    seed=seed,
)

In [45]:
sampler.train_data.shape

torch.Size([60, 1005])